In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_project.silver;

In [0]:
# Step 1: Read from Bronze
payers_df = spark.table("medical_project.bronze.payers")

In [0]:
# Step 2: Standardize Column Names
import re

def clean_column(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^\w]", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    col_name = col_name.strip("_")
    return col_name

payers_df = payers_df.toDF(*[clean_column(c) for c in payers_df.columns])


# Step 3: Remove Fivetran Metadata Columns
from pyspark.sql.functions import col

payers_df = payers_df.select([
    col(c) for c in payers_df.columns if not c.startswith("_")
])


# Step 4: Basic Data Type Fix (if needed)
payers_df = payers_df.withColumn("id", col("id").cast("string"))


# Step 5: Handle Null Values (Light)
payers_df = payers_df.fillna({
    "name": "unknown"
})


# Step 6: Remove Duplicates
payers_df = payers_df.dropDuplicates(["id"])


# Step 7: Standardize Text Columns
from pyspark.sql.functions import lower, trim

payers_df = payers_df.withColumn("name", lower(trim(col("name"))))


# Step 8: Filter Invalid Records
payers_df = payers_df.filter(col("id").isNotNull())

In [0]:
# Step 9: Write to Silver Layer
payers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.silver.payers")